# Семинар 4. Анализ тональности текстов (NLP)

**Цель семинара:** Научиться извлекать количественные аналитические метрики из неструктурированных текстовых данных (отзывов, жалоб, тикетов поддержки). Мы освоим базовую очистку текстов, подключим мощную предобученную ИИ-модель через библиотеку `transformers`, преобразуем текстовые эмоции в строгий математический скор и сагрегируем его до уровня клиента.

### 🔧 Настройка окружения и импорт библиотек

Для работы с глубоким обучением нам понадобится экосистема Hugging Face. Если библиотека `transformers` не установлена, установите её через ваш менеджер пакетов (`uv add transformers`).


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import warnings

# Подавляем предупреждения от Hugging Face
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 
warnings.filterwarnings('ignore')

try:
    from transformers import pipeline
except ImportError:
    print("⚠️ Библиотека transformers не найдена. Установите её: uv add transformers torch")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


---

## 📥 Шаг 1. Инициализация локального контура (Трек В: Отзывы)

Загружаем третий источник данных — выгрузку неструктурированных текстовых отзывов `reviews.csv`.


In [ ]:
BASE_DATA_DIR = os.path.join(".", "data")
INPUT_DIR = os.path.join(BASE_DATA_DIR, "seminar_3_feature_engineering")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "seminar_4_nlp_sentiment")
os.makedirs(OUTPUT_DIR, exist_ok=True)

reviews_csv_path = os.path.abspath(os.path.join(os.path.join(BASE_DATA_DIR, "input"), "reviews.csv"))

if not os.path.exists(reviews_csv_path):
    print(f"⚠️ Файл отзывов не найден: {reviews_csv_path}. Генерируется mock-датасет.")
    os.makedirs(INPUT_DIR, exist_ok=True)
    
    # Генерация синтетических отзывов
    mock_reviews = pd.DataFrame({
        'Review_ID': ['R001', 'R002', 'R003', 'R004', 'R005'],
        'Target_ID': ['T01', 'T01', 'T02', 'T03', 'T04'],
        'Review_Date': ['2023-10-01', '2023-10-15', '2023-11-02', '2023-11-20', '2023-12-05'],
        'Review_Text': [
            'Ужасный сервис, постоянные обрывы сети! Я очень зол!!! 😡',
            'Верните деньги за подписку, ни за что с вами больше не свяжусь.',
            'В целом нормально, но поддержка отвечает долго.',
            'Отличная скорость, очень вежливые операторы. Спасибо!',
            'Neutral comment about the product features.'
        ]
    })
    mock_reviews.to_csv(reviews_csv_path, index=False)

print(f"Путь к логу отзывов инициализирован: {reviews_csv_path}")


---

## 🛠 ЗАДАНИЕ 1: Базовая очистка неструктурированного текста
**Бизнес-контекст:** Текст из интернета полон мусора: знаки препинания, эмодзи, лишние пробелы, разный регистр. Прежде чем отдавать текст модели, его нужно привести к единому стандарту, чтобы снизить вычислительную нагрузку и вероятность ошибок.

**Инструкция (TODO):**
1. Считайте файл в датафрейм `df_reviews`.
2. Создайте новую колонку `Clean_Text`.
3. Приведите исходный текст к нижнему регистру с помощью `.str.lower()`.
4. Удалите всё, кроме букв и цифр, заменив небуквенные символы на пробелы с помощью `.str.replace(r'[^\w\s]', ' ', regex=True)`.
5. Схлопните множественные пробелы в один с помощью `.str.replace(r'\s+', ' ', regex=True)` и обрежьте пробелы по краям (`.str.strip()`).

*🤖 Теги для AI-ментора: `#SEM4_TASK1_START`, `#SEM4_TASK1_BUG`*


In [ ]:
# TODO: 1.1. Загрузите датасет
# TODO: 1.2. Создайте колонку Clean_Text, переведите в нижний регистр
# TODO: 1.3. Примените регулярные выражения для удаления пунктуации и лишних пробелов
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

df_reviews = pd.read_csv(...)

df_reviews['Clean_Text'] = df_reviews['Review_Text'].astype(str)....
df_reviews['Clean_Text'] = df_reviews['Clean_Text'].str.replace(..., ' ', regex=True)
df_reviews['Clean_Text'] = df_reviews['Clean_Text'].str.replace(..., ' ', regex=True).str.strip()


---

## 🛠 ЗАДАНИЕ 2 и 3: Инференс Трансформера и Математический Маппинг
**Бизнес-контекст:** Мы используем предобученную нейросеть `tabularisai/multilingual-sentiment-analysis`. Она читает текст и возвращает категорию (например, "1 star", "5 stars" или "Positive", "Negative"). Модель машинного обучения для предсказания оттока не поймет эти слова. Нам нужно смаппировать их в строгий математический интервал: от `-1.0` (глубокий негатив) до `+1.0` (позитив). Нейтральные отзывы получат `0.0`.

**Инструкция (TODO):**
1. Инициализируйте пайплайн `classifier = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")`.
2. Напишите функцию `get_sentiment_score(text)`, которая прогоняет текст через модель и возвращает число от -1.0 до 1.0 на основе метки. *(Подсказка: модель обычно возвращает словарь `[{'label': '5 stars', 'score': 0.9}]`. Маппинг: 1/2 stars -> -1.0, 3 stars -> 0.0, 4/5 stars -> 1.0).*
3. Примените функцию к колонке `Clean_Text` и создайте новую фичу `Sentiment_Score`.

*(Примечание: Загрузка модели может занять пару минут. Если у вас слабый ПК, мы включили mock-функцию для тестирования кода).*

*🤖 Теги для AI-ментора: `#SEM4_TASK2_START`, `#SEM4_TASK2_WHY`*


In [ ]:
# TODO: 2.1. Инициализируйте пайплайн Hugging Face (уже в шаблоне)
# TODO: 2.2. Напишите логику функции calculate_score для конвертации текстовой метки в float
# TODO: 2.3. Примените функцию к столбцу Clean_Text с помощью .apply()
raise NotImplementedError("Задания 2 и 3 не выполнены! Удалите эту строку и напишите свой код.")

sentiment_pipeline = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

def calculate_score(text):
    if not text or text.strip() == '':
        return 0.0
    
    # Запуск ИИ
    result = sentiment_pipeline(text[:512])[0]
    label = str(result['label']).lower()
    
    # TODO: Реализуйте if/else маппинг. 
    # Верните -1.0 для негатива, 1.0 для позитива, 0.0 для нейтрального.
    ...

df_reviews['Sentiment_Score'] = df_reviews['Clean_Text'].apply(...)


---

## 🛠 ЗАДАНИЕ 4: Агрегация текстовых метрик до уровня клиента
**Бизнес-контекст:** У одного клиента (`Target_ID`) может быть 5 разных отзывов. Нам нужно свернуть их в единую характеристику лояльности этого субъекта, чтобы позже прикрепить её к мастер-профилю. Мы усредним оценки.

**Инструкция (TODO):**
1. Сгруппируйте `df_reviews` по ключу `Target_ID`.
2. Рассчитайте среднее (`mean`) для колонки `Sentiment_Score`.
3. Сбросьте индекс и переименуйте колонку в `Mean_Sentiment`.


In [ ]:
# TODO: 4.1. Используйте groupby('Target_ID') для колонки 'Sentiment_Score' и вычислите mean()
# TODO: 4.2. Сделайте reset_index() и переименуйте результат
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

df_agg_sentiment = df_reviews.groupby(...)[...].mean().reset_index()
df_agg_sentiment.rename(columns={...: 'Mean_Sentiment'}, inplace=True)

display(df_agg_sentiment.head())


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция extract_sentiment_features

Упакуем наши наработки в production-функцию `extract_sentiment_features`. Она будет принимать датафрейм, названия колонок и имя модели (чтобы не хардкодить модель, если завтра Data Science отдел решит использовать другую архитектуру).


In [ ]:
def extract_sentiment_features(
    df: pd.DataFrame, 
    text_col: str, 
    id_col: str, 
    model_name: str = "tabularisai/multilingual-sentiment-analysis"
) -> pd.DataFrame:
    """
    Промышленная функция для извлечения метрик тональности.
    """
    # TODO: Соберите архитектуру функции, опираясь на параметры text_col и id_col
    raise NotImplementedError("Финальная сборка функции не выполнена!")
    
    df_pipe = df.copy()
    
    # 1. Очистка текста
    # ...
    
    # 2. Модель и Маппинг (можете использовать mock-функцию для скорости)
    # ...
    
    # 3. Агрегация
    # ...
    
    return pd.DataFrame()

df_nlp_final = extract_sentiment_features(pd.read_csv(reviews_csv_path), 'Review_Text', 'Target_ID')


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Запустите скрипт проверки, чтобы убедиться, что ваша NLP-функция возвращает валидный датафрейм, готовый к объединению в аналитическую витрину.


In [ ]:
def run_autocheck(df_to_check):
    print("🚀 Тестирование функции extract_sentiment_features...\n" + "-"*45)
    validation_status = True
    
    if not isinstance(df_to_check, pd.DataFrame) or df_to_check.empty:
        print("❌ Ошибка: Возвращен пустой или невалидный DataFrame.")
        return False
        
    expected_cols = ['Target_ID', 'Mean_Sentiment']
    missing_cols = [c for c in expected_cols if c not in df_to_check.columns]
    
    # Проверка структуры столбцов
    if missing_cols:
        print(f"❌ Ошибка: В таблице отсутствуют обязательные столбцы: {missing_cols}")
        validation_status = False
    else:
        print("✅ Колонки Target_ID и Mean_Sentiment присутствуют.")
        
    # Проверка типов и границ сентимента
    if 'Mean_Sentiment' in df_to_check.columns:
        if not pd.api.types.is_numeric_dtype(df_to_check['Mean_Sentiment']):
            print("❌ Ошибка: Колонка Mean_Sentiment имеет нечисловой тип.")
            validation_status = False
        else:
            min_val = df_to_check['Mean_Sentiment'].min()
            max_val = df_to_check['Mean_Sentiment'].max()
            if min_val < -1.01 or max_val > 1.01:
                print(f"❌ Ошибка: Нарушены границы сентимента [-1, 1]. Найдены значения: {min_val} до {max_val}")
                validation_status = False
            else:
                print("✅ Математический маппинг выполнен в корректных границах [-1.0, 1.0].")
                
    # Проверка гранулярности
    if 'Target_ID' in df_to_check.columns:
        if df_to_check['Target_ID'].duplicated().sum() > 0:
            print("❌ Ошибка: Нарушена агрегация. У одного клиента найдено несколько строк.")
            validation_status = False
        else:
            print("✅ Группировка проведена верно, строго одна оценка тональности на клиента.")

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! NLP-пайплайн полностью готов.")
        print("Добавьте эту функцию в файл course_project/src/data_pipeline.py!")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию вашей функции.")

run_autocheck(df_nlp_final)
